# 环节 01 · Tokenizer 分词（配套 Notebook）

> 配套长文：[环节01-Tokenizer分词详解.md](./环节01-Tokenizer分词详解.md)
> 定位：把长文里**手推的 BPE 合并表**跑成可改参数、可复现的代码。全部**纯 Python 标准库**实现（不用 numpy / torch）。

**怎么跑**

- 依赖：无。逐格 `Shift+Enter`；后面的格子依赖前面已执行的变量。
- 换成自己的语料：只改 §1 的 `CORPUS_RAW`，或 §4 的 `CORPUS_LONG`。
- 可选格（§7）需要 `pip install tiktoken transformers`，**没装只会打印安装提示，不影响其它格**。

**地图：本 Notebook ↔ 长文章节**

| 本 Notebook | 长文章节 | 验证什么 |
|---|---|---|
| §1 语料 | §3.2 | 中文无词边界，整段当“字流” |
| §2 字符级 BPE 训练 | §3.1 / §3.2 | 复现「人工 → 人工智 → 人工智能 → 学习」合并顺序 |
| §3 编码 / 解码 | §3.2 | 贪心最长匹配 + 无损还原 |
| §4 词表大小 vs 边际收益 | §6.1 | Zipf 长尾下“继续加词表”的收益为何迅速衰减 |
| §5 字节级 BPE | §4 / §5.3 | 高频片段被压缩、生僻字 / emoji 字节兜底（无 OOV） |
| §6 bytes/token 锚点 | §3.2b 尾注 | 英文 ≈4.1、中文基线 = 3 是怎么来的 |
| §7 真实 tokenizer 对照 | §5.3 / §5.4 | 同一句在不同词表下的 token 数 |


## 1. 语料：中文没有空格，整段当“字流”

用长文 §3.2 的微缩语料，**不做任何分词**（这就是“LLM 不用 jieba”的最小证据）：

```
学习人工智能 学习人工智能 未来 未来 人工智能
```


In [ ]:
CORPUS_RAW = "学习人工智能 学习人工智能 未来 未来 人工智能"
CORPUS = CORPUS_RAW.replace(" ", "")          # 中文无词边界：整段当字流

print("原始语料：", CORPUS_RAW)
print("字流    ：", " ".join(CORPUS))
print("初始符号：", sorted(set(CORPUS)), f"（{len(set(CORPUS))} 种）")
print("语料长度：", len(CORPUS), "字")


## 2. 从零实现 BPE（字符级）

BPE 只有两个动作：**数相邻对频次** → **合并最高频的那一对**，循环到词表预算耗尽。

- 起步符号：一个汉字 = 一个符号（字节级则是 256 个 UTF-8 字节，见 §5）；
- 真实训练在**全库**统计；这里把语料当成一条长序列，是“全库拼接”的简化版；
- 同频并列时按**首次出现顺序**取胜者（`dict` 保序，`max` 返回第一个最大值）；
- 真实工程不会每轮全库重扫，而是**增量更新受影响的邻接对 + 优先队列**维护最高频对（长文 §3.1 补注）。


In [ ]:
from collections import Counter


def get_stats(symbols, stats=None):
    """统计相邻符号对的频次。"""
    stats = Counter() if stats is None else stats
    for a, b in zip(symbols, symbols[1:]):
        stats[(a, b)] += 1
    return stats


def merge_pair(symbols, pair, new_symbol):
    """把序列里所有相邻的 pair 替换成 new_symbol（从左到右、不重叠）。"""
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i + 1] == pair[1]:
            out.append(new_symbol)
            i += 2
        else:
            out.append(symbols[i])
            i += 1
    return out


In [ ]:
def train_bpe(text, num_merges, verbose=True):
    """字符级 BPE：反复合并当前最高频的相邻对。

    返回 (merges, 最终符号序列, 每轮记录)。
    merges 的插入顺序 = 合并优先级，编码时按它贪心。
    """
    symbols = list(text)                  # 起步：一个汉字 = 一个初始符号
    merges = {}                           # (a, b) -> 合并出的新符号
    history = []
    for step in range(num_merges):
        stats = get_stats(symbols)
        if not stats:
            break
        pair, freq = max(stats.items(), key=lambda kv: kv[1])
        new_symbol = pair[0] + pair[1]
        symbols = merge_pair(symbols, pair, new_symbol)
        merges[pair] = new_symbol
        history.append((step + 1, pair, freq, new_symbol))
        if verbose:
            print(f"轮 {step + 1}: {pair[0]}+{pair[1]}（频次 {freq}）→ 「{new_symbol}」")
    return merges, symbols, history


merges, final_symbols, history = train_bpe(CORPUS, num_merges=6)
print("\n6 轮后的序列：", " ".join(map(str, final_symbols)))


**和长文 §3.2 手推表逐行对照**

| 轮次 | 长文表格 | 本 Notebook 实测 | 是否一致 |
|---|---|---|---|
| 1 | `人+工`(3) → `人工` | 同 | ✅ |
| 2 | `人工+智`(3) → `人工智` | 同 | ✅ |
| 3 | `人工智+能`(3) → `人工智能` | 同 | ✅ |
| 4 | `学+习`(2) → `学习` | 同 | ✅ |
| 5 | `未+来`(2) → `未来`（示意） | `学习+人工智能`(2) → `学习人工智能` | ⚠️ 见下 |

第 5 轮分岔的原因：`学习+人工智能` 的频次同样是 2，且它在语料里**出现得更靠前**——按“同频取先出现者”的规则，它先被合并。

> 这不是 bug，反而是重点：**BPE 的顺序完全由频次 + 语料顺序决定，没有人在挑词**。
> 把语料换一个顺序、换一份配比，词表就换一副面孔——这正是“词表反映训练语料”的微观证据（长文 §3.2b）。
> 长文表格第 5 行写 `未来` 是为了让示例看起来“更像词”，属示意。


In [ ]:
print("训出的合并规则（顺序即优先级）：")
for (a, b), new in merges.items():
    print(f"  {a} + {b}  →  {new}")

# 词表 = 初始字符 ∪ 所有合并产物；训完即冻结（长文 §7 工程红线）
vocab = sorted(set(CORPUS) | set(merges.values()), key=lambda t: (len(t), t))
print(f"\n最终词表（{len(vocab)} 项）：{vocab}")


## 3. 编码 / 解码：贪心最长匹配 + 无损还原

训练只发生一次、离线完成；**词表一旦冻结，编码就是纯查表**。

- 编码：从左到右，能匹配多长就切多长（长词优先）；
- 解码：token 直接拼接——这就是“可无损还原”；
- 语料外的字只能**逐字兜底** → 序列变长，这正是要上字节级 BPE 的原因（§5）。


In [ ]:
def build_vocab(text, merges):
    """最终词表：初始字符 + 所有合并产物。"""
    return set(text) | set(merges.values())


def encode(text, vocab):
    """贪心最长匹配：从左到右，能匹配多长就切多长。"""
    max_len = max(len(t) for t in vocab)
    tokens, i = [], 0
    while i < len(text):
        for L in range(min(max_len, len(text) - i), 0, -1):
            if text[i:i + L] in vocab:
                tokens.append(text[i:i + L])
                i += L
                break
        else:                             # 单字符必在初始字符集内，兜底不会触发
            tokens.append(text[i])
            i += 1
    return tokens


def decode(tokens):
    """子词 / 字符直接拼接即可无损还原。"""
    return "".join(tokens)


VOCAB = build_vocab(CORPUS, merges)

for s in ["人工智能未来已来", "未来已来", "未来人工智能", "人工智障"]:
    toks = encode(s, VOCAB)
    print(f"{s:>10} → {toks}   （{len(toks)} token，无损还原={decode(toks) == s}）")

print("\n词表外的字：", [c for c in "障" if c not in VOCAB],
      "（字符级只能逐字兜底 → 换字节级，见 §5）")


## 4. 词表大小 vs 边际收益：拐点怎么读（长文 §6.1）

把“跑多少轮 merge”当作“词表多大”的代理变量。有一个精确的等价关系：

> **某一轮合并省下的 token 数 = 这一对在本轮的频次**（每处出现都从 2 个符号变成 1 个符号）。

而 BPE 每轮只吃**当前频次最高**的那一对，所以“本轮省下多少”这条序列天然是**快速衰减**的。
衰减到“再加大词表也几乎换不来压缩”时，就是长文 §6.1 说的**收益拐点**：此后新增的名额全是低频长尾，只付 `|V| × d` 的开销。

要看到这个衰减，语料必须有**重尾（Zipf）分布**：少数词出现成千上万次，绝大多数词只出现一两次。真实预训练语料就是如此；为了在几百字内复现同样的形态，这里合成一段等价分布的语料（`seed` 固定、可复现，随意调比例重跑）。


In [ ]:
import random

# 真实语料的词频服从重尾（Zipf）分布：少数词极高频，绝大多数词只出现一两次。
# 这里合成一段等价分布的语料，好在几百字内复现“高频 → 长尾”的形态。
COMMON_WORDS = "模型 训练 数据 推理 词表 分词 算力 成本".split()
RARE_WORDS = ("注意力 位置编码 前馈网络 残差连接 归一化 梯度 采样 温度 掩码 缓存 "
              "吞吐 延迟 显存 并发 批处理 检索 向量 索引 切片 评测").split()

random.seed(7)
CORPUS_LONG = "".join(
    random.choice(RARE_WORDS) if random.random() < 0.12 else random.choice(COMMON_WORDS)
    for _ in range(600)
)

print(f"合成语料：{len(CORPUS_LONG)} 字（{len(COMMON_WORDS)} 个高频词 + "
      f"{len(RARE_WORDS)} 个长尾词，约 88:12 混合）")
print("开头 60 字：", CORPUS_LONG[:60])

CHECKPOINTS = (0, 1, 5, 10, 20, 40, 80, 150, 300)
MAX_ROUNDS = 300
n_bytes = len(CORPUS_LONG.encode("utf-8"))
base_tokens = len(CORPUS_LONG)

symbols = list(CORPUS_LONG)
rules = {}
saved = 0
freqs = []
rows = [(0, len(set(CORPUS_LONG)), None, 0, base_tokens)]       # 0 轮 = 纯字符，不合并
for step in range(1, MAX_ROUNDS + 1):
    stats = get_stats(symbols)
    if not stats:
        break
    pair, freq = max(stats.items(), key=lambda kv: kv[1])
    symbols = merge_pair(symbols, pair, pair[0] + pair[1])
    rules[pair] = pair[0] + pair[1]
    saved += freq                        # 这一轮省下的 token 数 == 该对频次
    freqs.append(freq)
    if step in CHECKPOINTS:
        vocab = set(CORPUS_LONG) | set(rules.values())
        rows.append((step, len(vocab), freq, saved, base_tokens - saved))

print(f"语料：{base_tokens} 字 / {n_bytes} 字节（不合并 = 每字 1 token，共 {base_tokens} token）\n")
print(f"{'轮次':>5} {'词表项数':>8} {'本轮省下':>8} {'累计省下':>8} {'剩余token':>9} {'bytes/token':>12}")
print("-" * 62)
for step, v, f, s, n_tok in rows:
    freq_txt = "-" if f is None else str(f)
    print(f"{step:>5} {v:>8} {freq_txt:>8} {s:>8} {n_tok:>9} {n_bytes / n_tok:>12.2f}")

print(f"\n“本轮省下 token 数”头 20 轮：{freqs[:20]}")
print("→ 前 8 轮稳定在 60~74（正好 8 个高频词的频次），第 9 轮起骤降到 10 上下（长尾词），")
print("  再往后每轮只省 1 个 —— 这就是边际收益递减，也是“长尾词条学不动”的量化形态。")
print("真实模型把这条曲线跑到上万轮 merge（128K+ 词表），拐点位置由语料构成和部署预算共同决定。")


## 5. 字节级 BPE：任意 Unicode 都能兜底（长文 §4 / §5.3）

把**起步符号从“字符”换成 UTF-8 字节（0~255）**，合并逻辑一字不改。主流模型（GPT / Qwen / DeepSeek）全是这条路线，所以**不存在 OOV**：

- 训练语料里**高频**的片段 → 被合并成 1 个 token（压缩收益在这里）；
- 没见过的生僻字 / emoji → 原样拆成几个字节 token（兜底），解码仍能无损还原。

> 关键：合并是**全局贪心**，每轮只吃掉“当前最高频”的那一对。
> 所以一个词能不能进词表，取决于它在语料里的**出现次数**，而不是它“看起来像不像一个词”——
> 这也是工业 tokenizer 要按语言配比重采样语料的原因（长文 §3.2b）。


In [ ]:
def train_bpe_bytes(text, num_merges, verbose=False):
    """字节级 BPE：起步符号是 UTF-8 字节（0~255），合并出的新 id 从 256 起。"""
    ids = list(text.encode("utf-8"))
    byte_map = {i: bytes([i]) for i in range(256)}     # id -> 字节串（解码用）
    merges = {}                                        # (id, id) -> 新 id
    next_id = 256
    for _ in range(num_merges):
        stats = Counter(zip(ids, ids[1:]))
        if not stats:
            break
        pair, freq = max(stats.items(), key=lambda kv: kv[1])
        byte_map[next_id] = byte_map[pair[0]] + byte_map[pair[1]]
        ids = merge_pair(ids, pair, next_id)
        merges[pair] = next_id
        if verbose:
            print(f"  {byte_map[pair[0]]!r} + {byte_map[pair[1]]!r} ×{freq} → id {next_id}")
        next_id += 1
    return merges, byte_map


def encode_bytes(text, merges, byte_map):
    """按“学到的先后顺序”反复合并：每次挑优先级最高的相邻对。"""
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = Counter(zip(ids, ids[1:]))
        pair = min(stats, key=lambda p: merges.get(p, 10 ** 9))
        if pair not in merges:
            break                                      # 剩下的部分就是字节兜底
        ids = merge_pair(ids, pair, merges[pair])
    return ids


def decode_bytes(ids, byte_map):
    """id → 字节串 → 文本；异常字节用 replace 兜住，永不抛错。"""
    return b"".join(byte_map[i] for i in ids).decode("utf-8", errors="replace")


# 中英混合的小语料：让「模型 / 训练 / 数据 / token」这类高频片段有机会进词表
TRAIN_TEXT = (
    "模型训练需要数据，模型推理需要算力。"
    "训练数据越多，模型越稳；推理成本越高，用量越要控制。"
    "模型把文本切成 token，token 越多，账单越高。"
    "The quick brown fox jumps over the lazy dog. "
    "Machine learning models learn patterns from data. "
    "Tokenization decides how many tokens a sentence costs. "
    "Rare characters such as 犇 and emoji never appear in the corpus. "
)

MERGE_B, BYTE_MAP = train_bpe_bytes(TRAIN_TEXT, num_merges=150)
print(f"训练语料：{len(TRAIN_TEXT)} 字 / {len(TRAIN_TEXT.encode('utf-8'))} 字节"
      f" → 学到 {len(MERGE_B)} 条合并规则\n")
print("最先学到的 5 条规则（id 从 256 起，每个 id 都对应一段真实字节）：")
for (a, b), nid in list(MERGE_B.items())[:5]:
    print(f"  {BYTE_MAP[a]!r} + {BYTE_MAP[b]!r} → id {nid} = {BYTE_MAP[nid]!r}")


In [ ]:
print(f"{'输入':>12} {'UTF-8字节':>10} {'token数':>8} {'bytes/token':>13}  无损还原")
print("-" * 68)
for s in ["模型", "训练", "数据", "token", "犇", "🚀", "模型🚀犇", "unseen"]:
    ids = encode_bytes(s, MERGE_B, BYTE_MAP)
    raw = len(s.encode("utf-8"))
    print(f"{s:>12} {raw:>10} {len(ids):>8} {raw / len(ids):>13.2f}  "
          f"{decode_bytes(ids, BYTE_MAP) == s}")

print("\n观察：")
print("  · 训练语料里反复出现的「模型 / 训练 / 数据」被合并成 1 个 token（6 字节 → 1 token）；")
print("  · 「犇」「🚀」等语料外字符回落为 3~4 个字节 token —— 序列变长，但不 OOV、不崩；")
print("  · 「模型🚀犇」混合场景：熟悉的片段照常压缩，陌生部分走字节兜底。")


## 6. bytes/token 锚点：数值怎么读（长文 §3.2b 尾注）

`bytes/token = 原文 UTF-8 字节数 ÷ token 数`，**只在同语言内比较才有意义**。

> ⚠️ 这里是 150 轮 merge 的玩具词表，绝对值会明显低于工业词表（128K+，上万轮 merge）。
> 要看的是三件事：**谁被合并了、谁走了兜底、数值随什么变**。


In [ ]:
EN = "The quick brown fox jumps over the lazy dog. " * 3
ZH = "模型训练需要数据，推理需要算力。" * 3

print(f"{'语言样例':>10} {'UTF-8字节':>10} {'token数':>8} {'bytes/token':>13} {'纯字节兜底':>11}")
print("-" * 72)
for name, text in [("英文", EN), ("中文", ZH)]:
    ids = encode_bytes(text, MERGE_B, BYTE_MAP)
    raw = len(text.encode("utf-8"))
    print(f"{name:>10} {raw:>10} {len(ids):>8} {raw / len(ids):>13.2f} {raw:>11}")

print("\n怎么读这几个数：")
print("  · “纯字节兜底”一列 = 完全不合并时的 token 数（每个字节 1 token），两列之差就是压缩收益；")
print("  · 中文若不合并 = 每字 3 token（汉字恰 3 字节）→ 合并后 token 数下降，bytes/token 才可能超过 3；")
print("  · 长文里的锚点：英文 ≈4.1、中文单字基线 = 3，都来自十万级词表；")
print("  · 数值还强烈依赖“词表语料 vs 测试文本”的配比 —— 本格词表以中文为主，英文表现自然就差。")


## 7. 可选：和真实 tokenizer 对照（长文 §5.3 / §5.4）

> 需要 `pip install tiktoken transformers`。没装 / 没网只会打印提示，前面几格照常有效。


In [ ]:
SAMPLES = ["人工智能", "The quick brown fox jumps over the lazy dog.", "犇🚀 混合 mixed 内容"]


def show(name, enc):
    print(f"\n== {name} ==")
    for s in SAMPLES:
        ids = enc(s)
        tail = " ..." if len(ids) > 12 else ""
        print(f"  {s!r:>42}  {len(ids):>3} token  {ids[:12]}{tail}")


try:
    import tiktoken
    show("tiktoken o200k_base（GPT-4o 词表）", tiktoken.get_encoding("o200k_base").encode)
except ImportError:
    print("未安装 tiktoken，跳过。安装：pip install tiktoken")

try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
    show("HF Qwen/Qwen2.5-0.5B（151K 词表，字节级 BPE）", tok.encode)
except ImportError:
    print("未安装 transformers，跳过。安装：pip install transformers")
except Exception as e:
    print("加载失败（多为网络问题），跳过：", type(e).__name__, e)

print("\n对照点：同一句话在不同词表下 token 数不同 —— “中文贵不贵”首先取决于词表。")
print("想量化自己的业务语料，直接照长文 §5.4 的四步验证跑一遍。")


## 8. 小结与面试自查

**代码侧一句话**：BPE = `数相邻对频次 → 合并最高频 → 记进词表`；训练离线一次、编码纯查表；把起步符号从字符换成 256 个字节，就同时拿到了“压缩”与“无 OOV 兜底”。

| 长文面试题 | 本 Notebook 的现场证据 |
|---|---|
| 为什么主流是子词而不是整词？ | §4：词表变大 token 数单调下降，但收益递减 |
| BPE 到底在比什么？ | §2 每轮打印的频次 + §2 第 5 轮分岔（同频看语料顺序） |
| 中文为什么要字节级？ | §3 语料外字只能逐字兜底 vs §5 字节兜底无 OOV |
| 为什么能处理任意 Unicode？ | §5 `犇 / 🚀` 全部无损还原 |
| 词表大小怎么定？ | §4 拐点 + §6 的 bytes/token 及格线 |
| 为什么不能换 tokenizer？ | §3 词表与 id 一一绑定，换了 id 语义全错位（长文 §7） |

**再往下走**：[环节 02 · Embedding 查表](./环节02-Embedding查表详解.md)——把这里产出的 id 序列变成向量。
